### Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### Load data

In [4]:
df = pd.read_csv("../data/simulated/hourly_energy_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

df.head()


,timestamp,electricity_price,temperature,production_load,historical_energy_usage
0,2026-01-01 00:00:00,0.915,-3.8,low,121.37
1,2026-01-01 01:00:00,0.977,-4.9,low,112.75
2,2026-01-01 02:00:00,0.917,-3.5,low,117.48
3,2026-01-01 03:00:00,0.937,-2.8,medium,150.58
4,2026-01-01 04:00:00,0.982,-1.2,medium,148.06


### Create time columns

In [9]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour

df.head()


,timestamp,electricity_price,temperature,production_load,historical_energy_usage,date,hour
0,2026-01-01 00:00:00,0.915,-3.8,low,121.37,2026-01-01,0
1,2026-01-01 01:00:00,0.977,-4.9,low,112.75,2026-01-01,1
2,2026-01-01 02:00:00,0.917,-3.5,low,117.48,2026-01-01,2
3,2026-01-01 03:00:00,0.937,-2.8,medium,150.58,2026-01-01,3
4,2026-01-01 04:00:00,0.982,-1.2,medium,148.06,2026-01-01,4


### Define process

In [13]:
PROCESS_DURATION_HOURS = 4
ENERGY_USUAGE_PER_HOUR_KWH = 120

STANDARD_START_HOUR = 8
STANDARD_END_HOUR = 12


### Function: find the cheapest 4-hrs window per day

In [11]:
def find_cheapest_window(day_df, duration_hours=4):
    day_df = day_df.sort_values("timestamp").reset_index(drop=True)

    windows = []

    for i in range(len(day_df) - duration_hours + 1):
        window = day_df.iloc[i:i + duration_hours]

        start_time = window["timestamp"].iloc[0]
        end_time = window["timestamp"].iloc[-1] + pd.Timedelta(hours=1)

        avg_price = window["electricity_price"].mean()
        total_price = window["electricity_price"].sum()

        windows.append({
            "date": start_time.date(),
            "optimized_start": start_time,
            "optimized_end": end_time,
            "optimized_avg_price": avg_price,
            "optimized_price_sum": total_price 
        })

    return pd.DataFrame(windows).sort_values("optimized_price_sum").iloc[0] 

### Find the cheapest window for each day

In [12]:
optimized_results = (
    df
    .groupby("date")
    .apply(find_cheapest_window, duration_hours=PROCESS_DURATION_HOURS)
    .reset_index(drop=True)
)

optimized_results.head()

,date,optimized_start,optimized_end,optimized_avg_price,optimized_price_sum
0,2026-01-01,2026-01-01 11:00:00,2026-01-01 15:00:00,0.85400,3.416
1,2026-01-02,2026-01-02 11:00:00,2026-01-02 15:00:00,0.82875,3.315
2,2026-01-03,2026-01-03 11:00:00,2026-01-03 15:00:00,0.73925,2.957
3,2026-01-04,2026-01-04 11:00:00,2026-01-04 15:00:00,0.76725,3.069
4,2026-01-05,2026-01-05 11:00:00,2026-01-05 15:00:00,0.84225,3.369


### Calculate costs for optimized running

In [14]:
optimized_results["optimized_cost"] = (
    optimized_results["optimized_price_sum"] * ENERGY_USUAGE_PER_HOUR_KWH
)

optimized_results.head()

,date,optimized_start,optimized_end,optimized_avg_price,optimized_price_sum,optimized_cost
0,2026-01-01,2026-01-01 11:00:00,2026-01-01 15:00:00,0.85400,3.416,409.92
1,2026-01-02,2026-01-02 11:00:00,2026-01-02 15:00:00,0.82875,3.315,397.80
2,2026-01-03,2026-01-03 11:00:00,2026-01-03 15:00:00,0.73925,2.957,354.84
3,2026-01-04,2026-01-04 11:00:00,2026-01-04 15:00:00,0.76725,3.069,368.28
4,2026-01-05,2026-01-05 11:00:00,2026-01-05 15:00:00,0.84225,3.369,404.28


### Standard running 08:00-12:00

In [15]:
standard_df = df[
    (df["hour"] >= STANDARD_START_HOUR) &
    (df["hour"] < STANDARD_END_HOUR) 
]

standard_results = (
    standard_df
    .groupby("date")
    .agg(
        standard_start=("timestamp", "min"),
        standard_end=("timestamp", "max"),
        standard_avg_price=("electricity_price", "mean"),
        standard_price_sum=("electricity_price", "sum"),
    )
    .reset_index()
)

standard_results["standard_end"] = standard_results["standard_end"] + pd.Timedelta(hours=1)

standard_results.head()

,date,standard_start,standard_end,standard_avg_price,standard_price_sum
0,2026-01-01,2026-01-01 08:00:00,2026-01-01 12:00:00,1.10225,4.409
1,2026-01-02,2026-01-02 08:00:00,2026-01-02 12:00:00,1.16675,4.667
2,2026-01-03,2026-01-03 08:00:00,2026-01-03 12:00:00,1.03650,4.146
3,2026-01-04,2026-01-04 08:00:00,2026-01-04 12:00:00,1.06325,4.253
4,2026-01-05,2026-01-05 08:00:00,2026-01-05 12:00:00,1.12625,4.505


### Calculate costs for standard running

In [16]:
standard_results["standard_cost"] = (
    standard_results["standard_price_sum"] * ENERGY_USUAGE_PER_HOUR_KWH
)

standard_results.head()

,date,standard_start,standard_end,standard_avg_price,standard_price_sum,standard_cost
0,2026-01-01,2026-01-01 08:00:00,2026-01-01 12:00:00,1.10225,4.409,529.08
1,2026-01-02,2026-01-02 08:00:00,2026-01-02 12:00:00,1.16675,4.667,560.04
2,2026-01-03,2026-01-03 08:00:00,2026-01-03 12:00:00,1.03650,4.146,497.52
3,2026-01-04,2026-01-04 08:00:00,2026-01-04 12:00:00,1.06325,4.253,510.36
4,2026-01-05,2026-01-05 08:00:00,2026-01-05 12:00:00,1.12625,4.505,540.60


### Compare standard against optimization

In [17]:
comparison = optimized_results.merge(
    standard_results,
    on="date",
    how="inner"
)

comparison["saving"] = comparison["standard_cost"] - comparison["optimized_cost"]

comparison["saving_percent"] = (
    comparison["saving"] / comparison["standard_cost"] * 100
)

comparison.head()

,date,optimized_start,optimized_end,optimized_avg_price,optimized_price_sum,optimized_cost,standard_start,standard_end,standard_avg_price,standard_price_sum,standard_cost,saving,saving_percent
0,2026-01-01,2026-01-01 11:00:00,2026-01-01 15:00:00,0.85400,3.416,409.92,2026-01-01 08:00:00,2026-01-01 12:00:00,1.10225,4.409,529.08,119.16,22.522114
1,2026-01-02,2026-01-02 11:00:00,2026-01-02 15:00:00,0.82875,3.315,397.80,2026-01-02 08:00:00,2026-01-02 12:00:00,1.16675,4.667,560.04,162.24,28.969359
2,2026-01-03,2026-01-03 11:00:00,2026-01-03 15:00:00,0.73925,2.957,354.84,2026-01-03 08:00:00,2026-01-03 12:00:00,1.03650,4.146,497.52,142.68,28.678244
3,2026-01-04,2026-01-04 11:00:00,2026-01-04 15:00:00,0.76725,3.069,368.28,2026-01-04 08:00:00,2026-01-04 12:00:00,1.06325,4.253,510.36,142.08,27.839172
4,2026-01-05,2026-01-05 11:00:00,2026-01-05 15:00:00,0.84225,3.369,404.28,2026-01-05 08:00:00,2026-01-05 12:00:00,1.12625,4.505,540.60,136.32,25.216426


### Conclussion

In [19]:
total_standard_cost = comparison["standard_cost"].sum()
total_optimized_cost = comparison["optimized_cost"].sum()
total_saving = comparison["saving"].sum()
total_saving_percent = total_saving / total_standard_cost * 100


print("Total cost standard running", round(total_standard_cost, 2))
print("Total cost optimized running", round(total_optimized_cost, 2))
print("Total saving", round(total_saving, 2))
print("Saving in percent", round(total_saving_percent, 2), "%")

Total cost standard running 47552.28
Total cost optimized running 35264.76
Total saving 12287.52
Saving in percent 25.84 %
